# 01 · EDGAR downloader (Data layer)

Downloads 8-K filings (main document + press-release exhibits) for a list of tickers, converts them to text,
flags likely **dividend** / **buyback** announcements, and builds a filing index.

Builds the ticker universe (180 dividend-paying S&P 400 / S&P 600 companies) and downloads their 2016–2026 8-Ks.

| Output (in the shared `Gen AI/Data/` folder) | What it is |
|---|---|
| `universe.csv` | the 180 companies (reused if it already exists) |
| `raw/edgar/<TICKER>/<accession_no>/` | original HTML: main 8-K + exhibit 99 press releases |
| `text/edgar/<TICKER>/<accession_no>/` | plain text of each document (Retrieval) |
| `index/filing_index.parquet` / `.csv` | one row per 8-K with dividend / buyback / exec-change flags |
| `index/dividend_coverage.csv` | dividend candidates vs dividends actually paid, per company |

**Setup:** set `USER_AGENT` below. The data folder comes from `$GENAI_DATA_DIR`, or in Colab from the mounted Drive.
Run top to bottom. Re-running is safe: anything already downloaded is skipped. A full run takes about 1.5–2 hours.

## 1. Config — edit this cell

In [ ]:
# Required by SEC: identify yourself (your name + email). Requests without it are blocked.
USER_AGENT = "Your Name your.email@example.com"

# Ticker universe: built automatically in section 4b
# (S&P 400 mid caps + S&P 600 small caps that pay dividends)
BUILD_UNIVERSE = True  # False = use the TICKERS list below instead
UNIVERSE_SIZE = 180  # half mid caps, half small caps
UNIVERSE_SEED = 42  # fixed so everyone gets the same list
EXCLUDE_SECTORS = ["Real Estate"]  # REIT payouts are mandated, so dividend changes say little
TICKERS = ["WSO", "UFPI", "CBRL"]  # only used if BUILD_UNIVERSE = False

START_DATE = "2016-01-01"
END_DATE = "2026-12-31"  # 2026 is a partial year

# Data folder (the shared Drive folder "Gen AI/Data/"):
#   - uses $GENAI_DATA_DIR if set (same variable as the repo's code),
#     e.g. your Google Drive for desktop path
#   - otherwise, in Colab, mounts Drive and uses the path below
#     (needs a "Gen AI" shortcut in My Drive)
COLAB_DATA_DIR = "/content/drive/MyDrive/Gen AI/Data"

# Optional: push the filing index to Google Sheets (Colab only). None = skip.
SHEET_URL = None  # e.g. "https://docs.google.com/spreadsheets/d/..."  or "new" to create one

# Which 8-Ks to keep, by 8-K item number (most 8-Ks are admin: votes, contracts, bylaws...)
NEEDED_ITEMS = [
    "2.02",
    "7.01",
    "8.01",
]  # earnings releases, Reg FD, other events: where dividends/buybacks appear
OPTIONAL_ITEMS = ["5.02"]  # executive/director changes: not used now, possible extra event type
KEEP_NO_ITEMS = True  # keep the rare 8-Ks with no item list, to be safe

MAX_REQUESTS_PER_SEC = 8  # SEC limit is 10; stay under it (applies across all download threads)
DOWNLOAD_WORKERS = 4  # parallel downloads
TEXT_WORKERS = (
    16  # parallel HTML -> text conversion (Drive is slow per file, so parallel helps a lot)
)
DOWNLOAD_LIMIT = None  # e.g. 50 to test quickly; None = all filings found

## 2. Setup

In [ ]:
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "requests",
        "pandas",
        "beautifulsoup4",
        "lxml",
        "pyarrow",
        "yfinance",
    ]
    + (["gspread"] if IN_COLAB else []),
    check=False,
)

In [ ]:
import os
import pathlib
import re
import threading
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning

# a few filings are XML; parsing them as HTML is fine
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

if "your.email@example.com" in USER_AGENT:
    raise ValueError("Set USER_AGENT in the config cell to your name and email (SEC requires it).")

MOUNTED = False
if os.environ.get("GENAI_DATA_DIR"):
    BASE = pathlib.Path(os.environ["GENAI_DATA_DIR"])
elif IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    MOUNTED = True
    BASE = pathlib.Path(COLAB_DATA_DIR)
else:
    raise RuntimeError(
        "Set GENAI_DATA_DIR to your local copy of the shared Drive folder 'Gen AI/Data/' "
        "(see README)."
    )

RAW = BASE / "raw" / "edgar"  # raw HTML per filing
TXT = BASE / "text" / "edgar"  # plain text per document (Retrieval uses this)
IDX = BASE / "index"  # filing index tables
for p in (RAW, TXT, IDX):
    p.mkdir(parents=True, exist_ok=True)
print("Saving to:", BASE.resolve())

## 3. Polite HTTP helper (rate limit + retries)

In [ ]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate"})
_lock, _next_slot = threading.Lock(), [0.0]


def _wait_turn():
    """Global rate limit shared by all threads."""
    with _lock:
        now = time.time()
        slot = max(now, _next_slot[0])
        _next_slot[0] = slot + 1 / MAX_REQUESTS_PER_SEC
    if slot > now:
        time.sleep(slot - now)


def get(url, as_json=False, retries=3):
    for attempt in range(retries):
        _wait_turn()
        r = SESSION.get(url, timeout=30)
        if r.status_code == 200:
            return r.json() if as_json else r.content
        if r.status_code == 403:
            raise RuntimeError(
                "403 Forbidden from SEC. Check USER_AGENT is set. If it still fails on Colab, "
                "SEC may be blocking cloud IPs - run this notebook on your laptop instead."
            )
        if r.status_code == 404:
            return None
        time.sleep(2**attempt)  # 429 / 5xx: back off and retry
    raise RuntimeError(f"Failed after {retries} tries: {url} (last status {r.status_code})")

## 4. Ticker → CIK

In [ ]:
tick = get("https://www.sec.gov/files/company_tickers.json", as_json=True)
CIK_OF = {v["ticker"].upper(): str(v["cik_str"]).zfill(10) for v in tick.values()}
NAME_OF = {v["ticker"].upper(): v["title"] for v in tick.values()}

missing = [t for t in TICKERS if t.upper() not in CIK_OF]
if missing:
    print("Not found in SEC ticker list (delisted or renamed?):", missing)
print({t: CIK_OF.get(t.upper()) for t in TICKERS})

## 4b. Build the ticker universe

Takes current S&P 400 (mid cap) and S&P 600 (small cap) members from Wikipedia, drops excluded sectors and
tickers SEC doesn't know, randomly samples (fixed seed), and keeps companies that paid **at least 8 dividends in 2016–2025**
(checked with yfinance). Saved to `universe.csv` and **reused on later runs**, so the whole team works from the same list.

> Limitation for the report: using *current* index members means companies that were delisted or dropped since 2016 are missing (survivorship bias).

In [ ]:
UNIVERSE_FILE = BASE / "universe.csv"


def wiki_constituents(url, bucket):
    import io

    html = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=30).text
    for t in pd.read_html(io.StringIO(html)):
        cols = {str(c).strip().lower(): c for c in t.columns}
        sym = next((cols[k] for k in ("symbol", "ticker", "ticker symbol") if k in cols), None)
        if sym is not None and len(t) > 300:
            sector = next((cols[k] for k in cols if "sector" in k), None)
            name = next((cols[k] for k in ("security", "company") if k in cols), None)
            return pd.DataFrame(
                {
                    "ticker": t[sym]
                    .astype(str)
                    .str.strip()
                    .str.replace(".", "-", regex=False)
                    .str.upper(),
                    "name": t[name] if name is not None else "",
                    "sector": t[sector] if sector is not None else "",
                    "bucket": bucket,
                }
            )
    raise ValueError(f"Constituent table not found on {url}")


def n_dividends(ticker):
    import yfinance as yf

    try:
        d = yf.Ticker(ticker).dividends
        return int(((d.index.year >= 2016) & (d.index.year <= 2025)).sum())
    except Exception:
        return 0


if not BUILD_UNIVERSE:
    print("Using TICKERS from config:", TICKERS)
elif UNIVERSE_FILE.exists():
    universe = pd.read_csv(UNIVERSE_FILE, dtype={"cik": str})
    TICKERS = universe["ticker"].tolist()
    print(
        f"Loaded existing universe ({len(TICKERS)} tickers) from {UNIVERSE_FILE}. "
        "Delete the file to rebuild."
    )
else:
    cands = pd.concat(
        [
            wiki_constituents("https://en.wikipedia.org/wiki/List_of_S%26P_400_companies", "mid"),
            wiki_constituents("https://en.wikipedia.org/wiki/List_of_S%26P_600_companies", "small"),
        ]
    )
    cands = cands[~cands["sector"].isin(EXCLUDE_SECTORS) & cands["ticker"].isin(CIK_OF)]
    cands = cands.drop_duplicates("ticker").sample(frac=1, random_state=UNIVERSE_SEED)  # shuffle

    per_bucket = UNIVERSE_SIZE // 2
    keep = []
    for bucket, grp in cands.groupby("bucket"):
        got = 0
        for row in grp.to_dict("records"):
            n = n_dividends(row["ticker"])
            if n >= 8:
                keep.append({**row, "n_dividends_2016_2025": n, "cik": CIK_OF[row["ticker"]]})
                got += 1
                if got >= per_bucket:
                    break
        print(f"{bucket}: kept {got} dividend payers")
    universe = pd.DataFrame(keep)
    universe.to_csv(UNIVERSE_FILE, index=False)
    TICKERS = universe["ticker"].tolist()
    print(f"Saved {len(TICKERS)} tickers to {UNIVERSE_FILE}")

universe.groupby(["bucket", "sector"]).size().unstack(0).fillna(0).astype(
    int
) if BUILD_UNIVERSE else None

## 5. List each company's 8-Ks

`submissions` JSON holds the most recent ~1,000 filings; older ones are in extra pages (`filings.files`), which are fetched only if needed.

> **Check:** SEC's `acceptanceDateTime` ends in `Z`, but verify on a few filings whether it is really UTC or US Eastern
> by comparing with the "Accepted" time on the filing's web page. This matters for `first_tradeable_date`.

In [ ]:
def _rows_from_block(block, ticker, cik):
    out = []
    n = len(block["accessionNumber"])
    for i in range(n):
        form = block["form"][i]
        date = block["filingDate"][i]
        if form in ("8-K", "8-K/A") and START_DATE <= date <= END_DATE:
            out.append(
                dict(
                    ticker=ticker,
                    cik=cik,
                    company_name=NAME_OF.get(ticker, ""),
                    form_type=form,
                    accession_no=block["accessionNumber"][i],
                    filing_date=date,
                    acceptance_datetime_raw=block["acceptanceDateTime"][i],
                    items=block["items"][i],
                    primary_doc=block["primaryDocument"][i],
                )
            )
    return out


def list_8ks(ticker):
    ticker = ticker.upper()
    cik = CIK_OF[ticker]
    sub = get(f"https://data.sec.gov/submissions/CIK{cik}.json", as_json=True)
    rows = _rows_from_block(sub["filings"]["recent"], ticker, cik)
    # older pages, only if they overlap our date range
    for f in sub["filings"].get("files", []):
        if f.get("filingTo", "9999") >= START_DATE:
            older = get(f"https://data.sec.gov/submissions/{f['name']}", as_json=True)
            rows += _rows_from_block(older, ticker, cik)
    return rows


filings = []
for t in TICKERS:
    if t.upper() in CIK_OF:
        r = list_8ks(t)
        print(f"{t}: {len(r)} 8-Ks")
        filings += r
filings_df = pd.DataFrame(filings).drop_duplicates("accession_no").reset_index(drop=True)
print("8-Ks found:", len(filings_df))

## 5b. Keep only the 8-K types we need (+ potentially useful)

Dividends and buybacks almost always appear under items **2.02** (earnings release), **7.01** (Reg FD) or **8.01** (other events).
**5.02** (executive changes) is kept as a possible extra event type. Everything else (shareholder votes, contracts, bylaws, debt...) is skipped.

In [ ]:
def item_set(s):
    return {x.strip() for x in str(s or "").split(",") if x.strip()}


KEEP = set(NEEDED_ITEMS) | set(OPTIONAL_ITEMS)
sets = filings_df["items"].apply(item_set)
mask = sets.apply(lambda it: bool(it & KEEP) or (KEEP_NO_ITEMS and not it))

print("Most common items (all 8-Ks):")
print(sets.explode().value_counts().head(12).to_string())
before = len(filings_df)
filings_df = filings_df[mask].reset_index(drop=True)
print(f"\nKept {len(filings_df)} of {before} 8-Ks ({len(filings_df) / before:.0%})")

## 6. Download each filing (main document + exhibit 99.x)

Dividend and buyback details are usually in **exhibit 99.1** (the press release), not the main 8-K body.
Exhibits are found from the filing's index page, using SEC's own **Type** column (`EX-99.1`, `EX-99.2`, ...),
so unusual filenames (e.g. `xex991.htm`, `exhibit991.htm`, `pressrelease.htm`) are not missed.

In [ ]:
EX99 = re.compile(r"ex[-_]?99|exhibit[-_]?99", re.I)  # fallback only


def filing_folder_url(cik, accession_no):
    return f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession_no.replace('-', '')}/"


def ex99_names(row, folder):
    """Exhibit 99.x filenames, read from the filing index page's Type column (cached to disk)."""
    page = folder / "filing_index.htm"
    if not page.exists():
        html = get(
            filing_folder_url(row["cik"], row["accession_no"]) + f"{row['accession_no']}-index.htm"
        )
        page.write_bytes(html or b"")
    soup = BeautifulSoup(page.read_bytes(), "lxml")
    names = []
    for tr in soup.select("table.tableFile tr"):
        tds = tr.find_all("td")
        if len(tds) >= 4 and tds[3].get_text(strip=True).upper().startswith("EX-99"):
            a = tds[2].find("a")
            if a and a.get("href"):
                name = a["href"].split("/")[-1]
                if name.lower().endswith((".htm", ".html", ".txt")):
                    names.append(name)
    if not names and not soup.select(
        "table.tableFile"
    ):  # page missing/odd -> fall back to filenames
        listing = get(
            filing_folder_url(row["cik"], row["accession_no"]) + "index.json", as_json=True
        ) or {"directory": {"item": []}}
        names = [
            it["name"]
            for it in listing["directory"]["item"]
            if EX99.search(it["name"]) and it["name"].lower().endswith((".htm", ".html", ".txt"))
        ]
    return names


def download_filing(row):
    folder = RAW / row["ticker"] / row["accession_no"]
    folder.mkdir(parents=True, exist_ok=True)
    base = filing_folder_url(row["cik"], row["accession_no"])
    docs = []

    # main document
    main = row["primary_doc"]
    if not (folder / main).exists():
        content = get(base + main)
        if content:
            (folder / main).write_bytes(content)
    if (folder / main).exists():
        docs.append(("main", main))

    # exhibits 99.x
    for name in ex99_names(row, folder):
        if name == main:
            continue
        if not (folder / name).exists():
            content = get(base + name)
            if content:
                (folder / name).write_bytes(content)
        if (folder / name).exists():
            docs.append(("ex99", name))
    return base, docs


def _download_one(row):
    try:
        base, docs = download_filing(row)
        return {
            "accession_no": row["accession_no"],
            "source_url": base,
            "docs": [d for _, d in docs],
            "n_ex99": sum(k == "ex99" for k, _ in docs),
            "error": None,
        }
    except Exception as e:
        return {
            "accession_no": row["accession_no"],
            "source_url": None,
            "docs": [],
            "n_ex99": 0,
            "error": str(e),
        }


todo = filings_df if DOWNLOAD_LIMIT is None else filings_df.head(DOWNLOAD_LIMIT)
results, t0 = [], time.time()
with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
    futures = [pool.submit(_download_one, row) for row in todo.to_dict("records")]
    for i, f in enumerate(as_completed(futures), 1):
        res = f.result()
        results.append(res)
        if res["error"] and "403" in res["error"]:
            pool.shutdown(wait=False, cancel_futures=True)
            raise RuntimeError(res["error"])
        if i % 250 == 0:
            print(f"{i}/{len(todo)} filings downloaded  ({(time.time() - t0) / 60:.1f} min)")
print("done:", len(results), "| errors:", sum(r["error"] is not None for r in results))

## 7. HTML → plain text (saved for Retrieval)

In [ ]:
def html_to_text(raw: bytes) -> str:
    soup = BeautifulSoup(raw, "lxml")
    for tag in soup(["script", "style"]):
        tag.decompose()
    text = soup.get_text("\n")
    text = re.sub(r"\xa0", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    return text.strip()


def text_for_filing(ticker, accession_no, docs):
    """Write one .txt per document and return the combined text (main + exhibits)."""
    parts = []
    out_dir = TXT / ticker / accession_no
    out_dir.mkdir(parents=True, exist_ok=True)
    for name in docs:
        src = RAW / ticker / accession_no / name
        dst = out_dir / (pathlib.Path(name).stem + ".txt")
        if not dst.exists():
            dst.write_text(html_to_text(src.read_bytes()), encoding="utf-8")
        parts.append(dst.read_text(encoding="utf-8"))
    return "\n\n".join(parts)

## 8. Flag likely dividend / buyback filings

A cheap keyword pre-filter so the LLM extraction step only sees candidates.
It is deliberately loose (false positives are fine; the LLM decides later).

In [ ]:
DIV_RE = re.compile(
    r"\b(declare[sd]?|declaration of)\b[^.]{0,120}\bdividends?\b"
    r"|\bdividends?\b[^.]{0,80}\bper share\b",
    re.I,
)
BUY_RE = re.compile(
    r"\b(share|stock) (repurchase|buyback|buy-back)s?\b|\brepurchase (program|authori[sz]ation)\b",
    re.I,
)


def snippet(text, m, width=160):
    s, e = max(0, m.start() - width), min(len(text), m.end() + width)
    return " ".join(text[s:e].split())


res_by_acc = {r["accession_no"]: r for r in results}


def _index_one(row):
    r = res_by_acc[row["accession_no"]]
    try:
        text = text_for_filing(row["ticker"], row["accession_no"], r["docs"]) if r["docs"] else ""
        err = r["error"]
    except Exception as e:
        text, err = "", f"text: {e}"
    d, b = DIV_RE.search(text), BUY_RE.search(text)
    items = item_set(row["items"])
    return {
        **row,
        "source_url": r["source_url"],
        "docs": ";".join(r["docs"]),
        "n_ex99": r["n_ex99"],
        "error": err,
        "n_chars": len(text),
        "is_div_candidate": bool(d),
        "is_buyback_candidate": bool(b),
        "is_exec_change": "5.02" in items,
        "div_snippet": snippet(text, d) if d else "",
        "buyback_snippet": snippet(text, b) if b else "",
    }


todo_rows = [row for row in todo.to_dict("records") if row["accession_no"] in res_by_acc]
rows, t0 = [], time.time()
with ThreadPoolExecutor(max_workers=TEXT_WORKERS) as pool:
    futures = [pool.submit(_index_one, row) for row in todo_rows]
    for i, f in enumerate(as_completed(futures), 1):
        rows.append(f.result())
        if i % 250 == 0:
            print(f"{i}/{len(todo_rows)} indexed  ({(time.time() - t0) / 60:.1f} min)")

index_df = pd.DataFrame(rows).sort_values(["ticker", "filing_date"]).reset_index(drop=True)
index_df["year"] = index_df["filing_date"].str[:4].astype(int)
index_df.to_parquet(IDX / "filing_index.parquet", index=False)
index_df.to_csv(IDX / "filing_index.csv", index=False)
print("Saved:", IDX / "filing_index.parquet")

## 9. Sanity checks — look at these before scaling up

In [ ]:
print("Filings:", len(index_df), "| errors:", index_df["error"].notna().sum())
print("With exhibit 99:", (index_df["n_ex99"] > 0).mean().round(2))
print()
summary = index_df.groupby("ticker").agg(
    filings=("accession_no", "count"),
    dividend_candidates=("is_div_candidate", "sum"),
    buyback_candidates=("is_buyback_candidate", "sum"),
    exec_changes=("is_exec_change", "sum"),
)
display(summary) if "display" in globals() else print(summary)

# 8-K item numbers on candidates: dividends usually 8.01 / 7.01 / 2.02 (earnings releases)
cand = index_df[index_df.is_div_candidate | index_df.is_buyback_candidate]
print("\nMost common items on candidates:")
print(cand["items"].str.split(",").explode().value_counts().head(10))

In [ ]:
# Read a few by eye: do the snippets really describe a dividend / buyback?
for _, r in (
    index_df[index_df.is_div_candidate]
    .sample(min(5, index_df.is_div_candidate.sum()), random_state=0)
    .iterrows()
):
    print(
        r.ticker, r.filing_date, r["items"], "\n  ", r.div_snippet[:300], "\n  ", r.source_url, "\n"
    )
for _, r in (
    index_df[index_df.is_buyback_candidate]
    .sample(min(5, index_df.is_buyback_candidate.sum()), random_state=0)
    .iterrows()
):
    print(
        r.ticker,
        r.filing_date,
        r["items"],
        "\n  ",
        r.buyback_snippet[:300],
        "\n  ",
        r.source_url,
        "\n",
    )

## 9c. Coverage check: are we catching each company's dividends?

Compares dividend candidates (8-Ks) with the number of dividends each company actually paid (from `universe.csv`).
A low ratio means that company announces dividends outside 8-Ks (press wire only) - note this as a data limitation.

In [ ]:
if BUILD_UNIVERSE and UNIVERSE_FILE.exists():
    u = pd.read_csv(UNIVERSE_FILE)[["ticker", "bucket", "n_dividends_2016_2025"]]
    c = (
        index_df[index_df["year"].between(2016, 2025)]
        .groupby("ticker")["is_div_candidate"]
        .sum()
        .rename("div_candidates")
    )
    cov = u.merge(c, on="ticker", how="left").fillna({"div_candidates": 0})
    cov["coverage"] = (cov["div_candidates"] / cov["n_dividends_2016_2025"]).clip(upper=1).round(2)
    print("Median coverage:", cov["coverage"].median())
    print("Companies with coverage >= 0.5:", int((cov["coverage"] >= 0.5).sum()), "of", len(cov))
    cov.to_csv(IDX / "dividend_coverage.csv", index=False)
    display(cov.sort_values("coverage").head(15)) if "display" in globals() else print(
        cov.sort_values("coverage").head(15)
    )

## 10. (Optional) Push the filing index to Google Sheets

In [ ]:
if SHEET_URL and IN_COLAB:
    import gspread
    from google.auth import default
    from google.colab import auth

    auth.authenticate_user()
    gc = gspread.authorize(default()[0])
    sh = gc.create("GenAI project data") if SHEET_URL == "new" else gc.open_by_url(SHEET_URL)
    # One tab per kind of table (not per year): filing_index now; events and gold_set later
    try:
        ws = sh.worksheet("filing_index")
    except gspread.WorksheetNotFound:
        ws = sh.add_worksheet("filing_index", rows=1000, cols=20)
    cols = [
        "year",
        "ticker",
        "company_name",
        "filing_date",
        "acceptance_datetime_raw",
        "form_type",
        "items",
        "accession_no",
        "is_div_candidate",
        "is_buyback_candidate",
        "is_exec_change",
        "div_snippet",
        "buyback_snippet",
        "source_url",
    ]
    out = index_df[cols].astype(str)
    ws.clear()
    ws.update([cols] + out.values.tolist())
    ws.freeze(rows=1)
    ws.set_basic_filter()  # filter by year / ticker / candidate flags in the sheet
    print("Sheet:", sh.url)
else:
    print("Skipped (set SHEET_URL and run in Colab).")

## 11. Make sure everything reaches Google Drive (run before closing the tab)

In [ ]:
if MOUNTED:
    from google.colab import drive

    drive.flush_and_unmount()
    print("All files synced to Drive. Re-run section 2 if you need Drive again in this session.")

## Next steps
- Run notebook 03 (prices) - it also builds `universe_quality.csv` from this filing index.
- Candidates are loose keyword matches, not events: the LLM extraction step decides which are real announcements.